In [ ]:
import os
import json
import re
import string

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================
# PARAMETERS
# ============================================

VOCAB_SIZE = 10000
MAX_LEN = 80

EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256

VALIDATION_SPLIT = 0.2
SEED = 42

LOAD_MODEL = False

BATCH_SIZE = 32

# Start with 5 epochs
EPOCHS = 5

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Parameters configured.")

In [ ]:
# ============================================
# DOWNLOAD WINE REVIEW DATASET
# ============================================

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "xsarinix/py_and_wine/master/"
    "winemag-data-130k-v2.json"
)

DATA_PATH = "/content/winemag-data-130k-v2.json"

!wget -q "{DATA_URL}" -O "{DATA_PATH}"

print("Download completed.")

In [ ]:
# ============================================
# CHECK DATASET
# ============================================

import os

print("File exists:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):

    size_mb = os.path.getsize(DATA_PATH) / (1024 ** 2)

    print(
        f"File size: {size_mb:.2f} MB"
    )

In [ ]:
# ============================================
# LOAD DATASET
# ============================================

with open(
    DATA_PATH,
    "r",
    encoding="utf-8"
) as json_data:

    wine_data = json.load(json_data)

print("Dataset loaded successfully!")
print(
    "Number of wine reviews:",
    len(wine_data)
)

In [ ]:
# ============================================
# INSPECT DATA
# ============================================

print(wine_data[10])

In [ ]:
# ============================================
# FILTER AND PREPARE DATA
# ============================================

filtered_data = []

for x in wine_data:

    if (
        x.get("country") is not None
        and x.get("province") is not None
        and x.get("variety") is not None
        and x.get("description") is not None
    ):

        text = (
            "wine review : "
            + str(x["country"])
            + " : "
            + str(x["province"])
            + " : "
            + str(x["variety"])
            + " : "
            + str(x["description"])
        )

        filtered_data.append(text)

n_wines = len(filtered_data)

print(
    f"{n_wines} wine reviews loaded"
)

In [ ]:
# ============================================
# EXAMPLE REVIEW
# ============================================

example = filtered_data[25]

print(example)

In [ ]:
# ============================================
# TEXT PREPROCESSING
# ============================================

def pad_punctuation(s):

    s = re.sub(
        f"([{string.punctuation}, '\\n'])",
        r" \1 ",
        s
    )

    s = re.sub(
        " +",
        " ",
        s
    )

    return s


text_data = [
    pad_punctuation(x)
    for x in filtered_data
]

print("Text preprocessing completed.")

In [ ]:
# ============================================
# DISPLAY PROCESSED REVIEW
# ============================================

example_data = text_data[25]

print(example_data)

In [ ]:
# ============================================
# CREATE TF DATASET
# ============================================

text_ds = (
    tf.data.Dataset
    .from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(
        1000,
        seed=SEED
    )
)

print("TensorFlow dataset created.")

In [ ]:
# ============================================
# TEXT VECTORIZATION
# ============================================

vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1
)

print(
    "TextVectorization layer created."
)

In [ ]:
# ============================================
# BUILD VOCABULARY
# ============================================

vectorize_layer.adapt(text_ds)

vocab = vectorize_layer.get_vocabulary()

print(
    "Vocabulary size:",
    len(vocab)
)

In [ ]:
# ============================================
# DISPLAY TOKEN MAPPINGS
# ============================================

for i, word in enumerate(vocab[:20]):

    print(
        f"{i}: {word}"
    )

In [ ]:
# ============================================
# CREATE TRAINING SET
# ============================================

def prepare_inputs(text):

    text = tf.expand_dims(
        text,
        -1
    )

    tokenized_sentences = (
        vectorize_layer(text)
    )

    x = tokenized_sentences[:, :-1]

    y = tokenized_sentences[:, 1:]

    return x, y


train_ds = text_ds.map(
    prepare_inputs,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

print("Training dataset ready.")

In [ ]:
# ============================================
# INSPECT TRAINING DATA
# ============================================

example_input_output = (
    train_ds
    .take(1)
    .get_single_element()
)

print(
    "Input shape:",
    example_input_output[0].shape
)

print(
    "Output shape:",
    example_input_output[1].shape
)

In [ ]:
# ============================================
# EXAMPLE INPUT
# ============================================

print(
    example_input_output[0][0].numpy()
)

In [ ]:
# ============================================
# EXAMPLE OUTPUT
# ============================================

print(
    example_input_output[1][0].numpy()
)

In [ ]:
# ============================================
# CAUSAL ATTENTION MASK
# ============================================

def causal_attention_mask(
    batch_size,
    n_dest,
    n_src,
    dtype
):

    i = tf.range(
        n_dest
    )[:, None]

    j = tf.range(
        n_src
    )

    m = (
        i >=
        j - n_src + n_dest
    )

    mask = tf.cast(
        m,
        dtype
    )

    mask = tf.reshape(
        mask,
        [1, n_dest, n_src]
    )

    mult = tf.concat(
        [
            tf.expand_dims(
                batch_size,
                -1
            ),
            tf.constant(
                [1, 1],
                dtype=tf.int32
            )
        ],
        0
    )

    return tf.tile(
        mask,
        mult
    )

In [ ]:
# ============================================
# VISUALIZE CAUSAL MASK
# ============================================

mask = causal_attention_mask(
    1,
    10,
    10,
    dtype=tf.int32
)

plt.figure(
    figsize=(6, 6)
)

plt.imshow(
    mask[0].numpy(),
    cmap="gray"
)

plt.title(
    "Causal Attention Mask"
)

plt.xlabel(
    "Source Position"
)

plt.ylabel(
    "Destination Position"
)

plt.show()

In [ ]:
# ============================================
# TRANSFORMER BLOCK
# ============================================

class TransformerBlock(
    layers.Layer
):

    def __init__(
        self,
        num_heads,
        key_dim,
        embed_dim,
        ff_dim,
        dropout_rate=0.1
    ):

        super().__init__()

        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.attn = (
            layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=key_dim,
                output_shape=embed_dim
            )
        )

        self.dropout_1 = layers.Dropout(
            dropout_rate
        )

        self.ln_1 = layers.LayerNormalization(
            epsilon=1e-6
        )

        self.ffn_1 = layers.Dense(
            ff_dim,
            activation="relu"
        )

        self.ffn_2 = layers.Dense(
            embed_dim
        )

        self.dropout_2 = layers.Dropout(
            dropout_rate
        )

        self.ln_2 = layers.LayerNormalization(
            epsilon=1e-6
        )

    def call(
        self,
        inputs,
        training=None
    ):

        input_shape = tf.shape(
            inputs
        )

        batch_size = input_shape[0]

        seq_len = input_shape[1]

        causal_mask = (
            causal_attention_mask(
                batch_size,
                seq_len,
                seq_len,
                tf.bool
            )
        )

        attention_output, attention_scores = (
            self.attn(
                inputs,
                inputs,
                attention_mask=causal_mask,
                return_attention_scores=True,
                training=training
            )
        )

        attention_output = (
            self.dropout_1(
                attention_output,
                training=training
            )
        )

        out1 = self.ln_1(
            inputs + attention_output
        )

        ffn_1 = self.ffn_1(
            out1
        )

        ffn_2 = self.ffn_2(
            ffn_1
        )

        ffn_output = (
            self.dropout_2(
                ffn_2,
                training=training
            )
        )

        return (
            self.ln_2(
                out1 + ffn_output
            ),
            attention_scores
        )

    def get_config(self):

        config = super().get_config()

        config.update({
            "key_dim": self.key_dim,
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate
        })

        return config

In [ ]:
# ============================================
# TOKEN + POSITION EMBEDDING
# ============================================

class TokenAndPositionEmbedding(
    layers.Layer
):

    def __init__(
        self,
        max_len,
        vocab_size,
        embed_dim
    ):

        super().__init__()

        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

        self.token_emb = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.pos_emb = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, x):

        maxlen = tf.shape(x)[-1]

        positions = tf.range(
            start=0,
            limit=maxlen,
            delta=1
        )

        positions = self.pos_emb(
            positions
        )

        x = self.token_emb(
            x
        )

        return x + positions

    def get_config(self):

        config = super().get_config()

        config.update({
            "max_len": self.max_len,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim
        })

        return config

In [ ]:
# ============================================
# BUILD GPT
# ============================================

inputs = layers.Input(
    shape=(None,),
    dtype=tf.int32
)

x = TokenAndPositionEmbedding(
    MAX_LEN,
    VOCAB_SIZE,
    EMBEDDING_DIM
)(inputs)

x, attention_scores = TransformerBlock(
    N_HEADS,
    KEY_DIM,
    EMBEDDING_DIM,
    FEED_FORWARD_DIM
)(x)

outputs = layers.Dense(
    VOCAB_SIZE,
    activation="softmax"
)(x)

gpt = models.Model(
    inputs=inputs,
    outputs=[
        outputs,
        attention_scores
    ]
)

gpt.compile(
    optimizer="adam",
    loss=[
        losses.SparseCategoricalCrossentropy(),
        None
    ]
)

print("GPT model created successfully.")

In [ ]:
# ============================================
# MODEL SUMMARY
# ============================================

gpt.summary()

In [ ]:
# ============================================
# TEST MODEL
# ============================================

test_x = example_input_output[0][:2]

test_output = gpt.predict(
    test_x,
    verbose=0
)

print(
    "Prediction shape:",
    test_output[0].shape
)

print(
    "Attention shape:",
    test_output[1].shape
)

In [ ]:
# ============================================
# CREATE DIRECTORIES
# ============================================

os.makedirs(
    "/content/gpt_checkpoint",
    exist_ok=True
)

os.makedirs(
    "/content/gpt_logs",
    exist_ok=True
)

os.makedirs(
    "/content/gpt_models",
    exist_ok=True
)

print("Directories created.")

In [ ]:
# ============================================
# TEXT GENERATOR
# ============================================

class TextGenerator(
    callbacks.Callback
):

    def __init__(
        self,
        index_to_word,
        top_k=10
    ):

        super().__init__()

        self.index_to_word = index_to_word

        self.word_to_index = {
            word: index
            for index, word
            in enumerate(index_to_word)
        }

        self.top_k = top_k

    def sample_from(
        self,
        probs,
        temperature
    ):

        probs = np.asarray(
            probs
        ).astype("float64")

        # Temperature
        probs = (
            probs ** (1.0 / temperature)
        )

        probs = (
            probs /
            np.sum(probs)
        )

        token = np.random.choice(
            len(probs),
            p=probs
        )

        return token, probs

    def generate(
        self,
        start_prompt,
        max_tokens=80,
        temperature=1.0
    ):

        start_tokens = [
            self.word_to_index.get(
                x,
                1
            )
            for x in start_prompt.split()
        ]

        sample_token = None

        info = []

        while (
            len(start_tokens)
            < max_tokens
            and sample_token != 0
        ):

            x = np.array(
                [start_tokens]
            )

            y, att = self.model.predict(
                x,
                verbose=0
            )

            sample_token, probs = (
                self.sample_from(
                    y[0][-1],
                    temperature
                )
            )

            info.append({
                "prompt": start_prompt,
                "word_probs": probs,
                "atts": att[0, :, -1, :]
            })

            start_tokens.append(
                sample_token
            )

            if sample_token < len(
                self.index_to_word
            ):

                next_word = (
                    self.index_to_word[
                        sample_token
                    ]
                )

            else:

                next_word = ""

            start_prompt = (
                start_prompt
                + " "
                + next_word
            )

        print(
            "\nGenerated text:"
        )

        print(
            "--------------------------------"
        )

        print(
            start_prompt
        )

        print(
            "--------------------------------"
        )

        return info

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        print(
            f"\n--- Epoch {epoch + 1} ---"
        )

        self.generate(
            "wine review",
            max_tokens=50,
            temperature=1.0
        )

In [ ]:
# ============================================
# TRAINING CALLBACKS
# ============================================

model_checkpoint_callback = (
    callbacks.ModelCheckpoint(
        filepath=(
            "/content/gpt_checkpoint/"
            "gpt.weights.h5"
        ),
        save_weights_only=True,
        save_freq="epoch",
        verbose=1
    )
)

tensorboard_callback = (
    callbacks.TensorBoard(
        log_dir="/content/gpt_logs"
    )
)

text_generator = TextGenerator(
    vocab
)

print("Callbacks ready.")

In [ ]:
# ============================================
# TRAIN GPT
# ============================================

history = gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[
        model_checkpoint_callback,
        tensorboard_callback,
        text_generator
    ]
)

In [ ]:
# ============================================
# TRAINING LOSS
# ============================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history.history["loss"],
    marker="o"
)

plt.title(
    "GPT Training Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.grid(True)

plt.show()

In [ ]:
# ============================================
# GENERATE USA WINE REVIEW
# ============================================

info_usa = text_generator.generate(
    "wine review : us",
    max_tokens=80,
    temperature=1.0
)

In [ ]:
# ============================================
# GENERATE ITALIAN WINE REVIEW
# ============================================

info_italy = text_generator.generate(
    "wine review : italy",
    max_tokens=80,
    temperature=0.5
)

In [ ]:
# ============================================
# GENERATE GERMAN WINE REVIEW
# ============================================

info_germany = text_generator.generate(
    "wine review : germany",
    max_tokens=80,
    temperature=0.5
)

In [ ]:
# ============================================
# DISPLAY TOP WORD PROBABILITIES
# ============================================

def print_probs(
    info,
    vocab,
    top_k=5
):

    for item in info:

        print(
            "\nPROMPT:"
        )

        print(
            item["prompt"]
        )

        word_probs = item[
            "word_probs"
        ]

        top_indices = np.argsort(
            word_probs
        )[::-1][:top_k]

        print(
            "\nTop predictions:"
        )

        for idx in top_indices:

            probability = (
                word_probs[idx] * 100
            )

            print(
                f"{vocab[idx]}: "
                f"{probability:.2f}%"
            )

        print(
            "----------------------"
        )

In [ ]:
# ============================================
# ANALYZE PREDICTIONS
# ============================================

print_probs(
    info_germany,
    vocab,
    top_k=5
)

In [ ]:
# ============================================
# TEMPERATURE COMPARISON
# ============================================

prompt = "wine review : italy"

print("\n==============================")
print("TEMPERATURE = 0.2")
print("==============================")

text_generator.generate(
    prompt,
    max_tokens=60,
    temperature=0.2
)

print("\n==============================")
print("TEMPERATURE = 0.5")
print("==============================")

text_generator.generate(
    prompt,
    max_tokens=60,
    temperature=0.5
)

print("\n==============================")
print("TEMPERATURE = 1.0")
print("==============================")

text_generator.generate(
    prompt,
    max_tokens=60,
    temperature=1.0
)

In [ ]:
# ============================================
# CUSTOM WINE REVIEW GENERATOR
# ============================================

def generate_wine_review(
    country,
    max_tokens=80,
    temperature=0.7
):

    prompt = (
        "wine review : "
        + country.lower()
    )

    print(
        f"\nGenerating review for: "
        f"{country}"
    )

    return text_generator.generate(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature
    )

In [ ]:
generate_wine_review(
    "France"
)

In [ ]:
# ============================================
# SAVE GPT MODEL
# ============================================

MODEL_PATH = (
    "/content/gpt_models/"
    "wine_gpt.keras"
)

gpt.save(
    MODEL_PATH
)

print(
    "Model saved successfully!"
)

print(
    MODEL_PATH
)